# Notebook 3 — Candidate docking, pose QC, PLIP integration

This revision preserves the existing `ddr1_config.py` path definitions, project folders,
input filenames, and output filenames. It strengthens tool checks, receptor-file integrity,
pose atom-order validation, symmetry-aware direct RMSD, and optional PLIP handling.

Important interpretation:
- Docking poses are compared in the receptor coordinate frame; no coordinate superposition is used for pose reproducibility.
- Symmetry correction changes atom correspondence only; it does not translate or rotate poses.
- PLIP is optional for the docking run. If its executable is unavailable, docking and pose QC continue, and PLIP outputs are explicitly marked unavailable.
- Vina scores and tier thresholds are prioritization heuristics, not experimental potency estimates.


In [1]:
#Installing the required packages

%pip install meeko==0.8.0
!apt-get update -qq
!apt-get install -y autodock-vina
%pip install rdkit
%pip -q install biopython
%pip install gemmi
%pip install prody
%pip install -q plip
import Bio
print("Biopython version:", Bio.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.3 MB/s eta 0:00:00a 0:00:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-filesystem1.83.0 libboost-program-options1.83.0
  libboost-thread1.83.0
Suggested packages:
  autodock autogrid
The following NEW packages will be installed:
  autodock-vina libboost-filesystem1.83.0 libboost-program-options1.83.0
  libboost-thread1.83.0
0 upgraded, 4 newly installed, 0 to remove and 27 not upgraded.
Need to get 1,166 kB of archives.
After this operation, 7,820 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libboost-filesystem1.83.0 amd64 1.83.0-2.1ubuntu3.2 [284 kB]
Get:2 http://arc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 22.8 MB/s eta 0:00:0000:0100:01
Biopython version: 1.88


In [2]:
# CELL 1 — Config, paths, environment
import sys, json, shutil, subprocess, re, time, hashlib, os
from pathlib import Path
from datetime import datetime, timezone

PROJECT_NAME = "DDR1_Workflow"
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

_search = [Path("/content/drive/MyDrive") / PROJECT_NAME, Path("/content"),
           Path.home(), Path.cwd(), *Path.cwd().parents]
for _c in _search:
    if (_c / "ddr1_config.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break
else:
    raise FileNotFoundError(
        "ddr1_config.py not found. Expected it under the existing project folder "
        f"or one of the searched locations: {[str(p) for p in _search]}"
    )
sys.modules.pop("ddr1_config", None)

from ddr1_config import (setup_paths, sha256_file, write_ligand_pdb,
                         direct_rmsd, kabsch_rmsd, strip_salts_and_neutralize)

PATHS = setup_paths(project_name=PROJECT_NAME)
ML_RESULTS = PATHS["ML_RESULTS"]
NB2_RECEPTOR_PDBQT = PATHS["REDOCKING_RECEPTOR"] / "5SB1_receptor.pdbqt"
NB2_RECEPTOR_PDB = PATHS["REDOCKING_RECEPTOR"] / "5SB1_receptor.pdb"
NB2_BOX_JSON = PATHS["REDOCKING_REPORTS"] / "redocking_box.json"

MANUSCRIPT = PATHS["CANDIDATE_MANUSCRIPT"]
LIGAND_PREP = PATHS["CANDIDATE_LIGAND_PREP"]
LIGAND_PDBQT_DIR = PATHS["CANDIDATE_LIGAND_PDBQT"]
DOCKING_DIR = PATHS["CANDIDATE_DOCKING"]
POSE_REPRO_DIR = PATHS["CANDIDATE_POSE_REPRO"]
REPRESENTATIVE_DIR = PATHS["CANDIDATE_REPRESENTATIVE"]
COMPLEX_DIR = PATHS["CANDIDATE_COMPLEXES"]
PLIP_DIR = PATHS["CANDIDATE_PLIP"]
REPORT_DIR = PATHS["CANDIDATE_REPORTS"]

# Keep the established input filename unchanged.
CANDIDATE_FILE = ML_RESULTS / "DDR1_DrugBank_Top10_strict.csv"
if not CANDIDATE_FILE.exists():
    raise FileNotFoundError(f"{CANDIDATE_FILE} not found. Run Notebook 1 first.")
for _p in (NB2_RECEPTOR_PDBQT, NB2_RECEPTOR_PDB, NB2_BOX_JSON):
    if not _p.exists():
        raise FileNotFoundError(f"Required Notebook 2 output is missing: {_p}. Run Notebook 2 first.")

BOX = json.loads(NB2_BOX_JSON.read_text())
_required_box_keys = ("center_x", "center_y", "center_z", "size_x", "size_y", "size_z")
_missing_box = [k for k in _required_box_keys if k not in BOX]
if _missing_box:
    raise KeyError(f"{NB2_BOX_JSON} lacks required docking-box keys: {_missing_box}")
if any(float(BOX[k]) <= 0 for k in ("size_x", "size_y", "size_z")):
    raise ValueError(f"Docking-box dimensions must be positive: {BOX}")
print("Box from NB2:", BOX)

MEEKO_FOUND = shutil.which("mk_prepare_ligand.py")
VINA_FOUND = shutil.which("vina")
MEEKO_LIGAND = Path(MEEKO_FOUND or "/usr/local/bin/mk_prepare_ligand.py")
VINA = Path(VINA_FOUND or "/usr/local/bin/vina")
PLIP_EXE = shutil.which("plip")
PLIP_AVAILABLE = PLIP_EXE is not None

if not MEEKO_LIGAND.exists():
    raise FileNotFoundError(
        "Meeko executable mk_prepare_ligand.py was not found. Install/configure Meeko "
        "before running ligand preparation; no project paths were changed."
    )
if not VINA.exists():
    raise FileNotFoundError(
        "AutoDock Vina executable was not found. Install/configure Vina before docking."
    )
if PLIP_AVAILABLE:
    print("PLIP executable:", PLIP_EXE)
else:
    print("WARNING: PLIP executable is not on PATH. Docking will continue; interaction "
          "profiling will be marked SKIPPED_NO_PLIP. Install PLIP and rerun Cells 9–12 "
          "to generate interaction tables.")

EXHAUSTIVENESS = 32
NUM_MODES = 20
ENERGY_RANGE = 4.0
CPU = 2
SEEDS = [7, 42, 2026]
POSE_CONSISTENCY_THRESHOLD = 2.0

print("Candidates:", CANDIDATE_FILE)
print("Vina:", VINA, "| Meeko ligand:", MEEKO_LIGAND, "| PLIP:", PLIP_EXE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : COLAB
Project Root: /content/drive/MyDrive/DDR1_Workflow
Directories : 30
Box from NB2: {'center_x': 22.764719009399414, 'center_y': -2.076000213623047, 'center_z': -1.1666250228881836, 'size_x': 22.0, 'size_y': 22.0, 'size_z': 22.0, 'derivation': 'centroid of crystallographic 1JI in 5SB1 chain A'}
PLIP executable: /usr/local/bin/plip
Candidates: /content/drive/MyDrive/DDR1_Workflow/ml_model/results/DDR1_DrugBank_Top10_strict.csv
Vina: /usr/bin/vina | Meeko ligand: /usr/local/bin/mk_prepare_ligand.py | PLIP: /usr/local/bin/plip


In [3]:
# CELL 2 — Reuse NB2 receptor with independent SHA256 verification
import shutil as _sh
RECEPTOR_DIR = PATHS["CANDIDATE"] / "receptor"
RECEPTOR_DIR.mkdir(parents=True, exist_ok=True)
RECEPTOR_PDBQT = RECEPTOR_DIR / "5SB1_receptor.pdbqt"
RECEPTOR_PDB = RECEPTOR_DIR / "5SB1_receptor.pdb"

# Copy/check each file independently: a correct PDBQT must not mask a missing or stale PDB.
for _source, _target in ((NB2_RECEPTOR_PDBQT, RECEPTOR_PDBQT),
                         (NB2_RECEPTOR_PDB, RECEPTOR_PDB)):
    _source_hash = sha256_file(_source)
    if not _target.exists() or sha256_file(_target) != _source_hash:
        _sh.copy2(_source, _target)
    _target_hash = sha256_file(_target)
    if _target_hash != _source_hash:
        raise RuntimeError(f"Receptor copy verification failed: {_target}")
    print(f"Verified {_target.name}: SHA256={_target_hash}")

if RECEPTOR_PDBQT.stat().st_size == 0 or RECEPTOR_PDB.stat().st_size == 0:
    raise RuntimeError("Receptor PDB/PDBQT file is empty.")


Verified 5SB1_receptor.pdbqt: SHA256=f52b93259515a7802d605245c78127c886b24163ab040539cd873fb7ed588a74
Verified 5SB1_receptor.pdb: SHA256=8731d2e67f6432ad13e98bf49ea51ac4894656f9b4cd8d00fa3aeda104f08e49


In [4]:
# CELL 3 — Load candidates, prepare 3D SDFs
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")

candidates = pd.read_csv(CANDIDATE_FILE)
name_col = next((c for c in ("Name", "Drug_Name", "name", "drug_name") if c in candidates.columns), None)
smiles_col = next((c for c in ("SMILES", "smiles", "Canonical_SMILES", "canonical_smiles")
                   if c in candidates.columns), None)
if name_col is None or smiles_col is None:
    raise KeyError(f"Could not find name/SMILES in {CANDIDATE_FILE}")
candidates = candidates.rename(columns={name_col: "Name", smiles_col: "SMILES"})
candidates = candidates[candidates["Name"].astype(str).str.strip().ne("")]
candidates = candidates.drop_duplicates(subset=["Name"]).reset_index(drop=True)
print(f"Loaded {len(candidates)} candidates")

LIGAND_PREP.mkdir(parents=True, exist_ok=True)
prep_records = []
for _, row in candidates.iterrows():
    name = str(row["Name"]).strip()
    canon = strip_salts_and_neutralize(str(row["SMILES"]))
    if canon is None:
        prep_records.append({"Name": name, "Status": "SMILES_PARSE_FAILED"}); continue
    mol = Chem.MolFromSmiles(canon)
    mol_h = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol_h, randomSeed=20260914, useRandomCoords=True) != 0:
        prep_records.append({"Name": name, "Status": "EMBED_FAILED"}); continue
    try:
        mmff_ok = AllChem.MMFFHasAllMoleculeParams(mol_h)
    except Exception:
        mmff_ok = False
    if mmff_ok:
        AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant="MMFF94", maxIters=500)
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    sdf_path = LIGAND_PREP / f"{safe}.sdf"
    w = Chem.SDWriter(str(sdf_path)); w.write(mol_h); w.close()
    prep_records.append({"Name": name, "Canonical_SMILES": canon,
                         "MMFF_applicable": mmff_ok, "SDF": str(sdf_path),
                         "Status": "PREPARED"})

prepared_df = pd.DataFrame(prep_records)
prepared_df = prepared_df[prepared_df["Status"] == "PREPARED"].reset_index(drop=True)
prepared_df.to_csv(LIGAND_PREP / "prepared_candidates.csv", index=False)
print(f"Prepared: {len(prepared_df)}")
if prepared_df.empty:
    raise RuntimeError("No candidates prepared")

Loaded 6 candidates
Prepared: 6


In [5]:
# CELL 4 — Meeko ligand PDBQT
LIGAND_PDBQT_DIR.mkdir(parents=True, exist_ok=True)
rows = []
for _, row in prepared_df.iterrows():
    name = row["Name"]; safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))
    sdf_path = Path(row["SDF"]); out = LIGAND_PDBQT_DIR / f"{safe}.pdbqt"
    if out.exists() and out.stat().st_size > 100:
        n = sum(1 for L in out.read_text(errors="replace").splitlines() if L.startswith(("ATOM", "HETATM")))
        if n > 0:
            rows.append({"Name": name, "PDBQT": str(out), "Atom_count": n, "Status": "REUSED"}); continue
    cmd = [str(MEEKO_LIGAND), "-i", str(sdf_path), "-o", str(out)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0 and out.exists():
        n = sum(1 for L in out.read_text(errors="replace").splitlines() if L.startswith(("ATOM", "HETATM")))
        rows.append({"Name": name, "PDBQT": str(out), "Atom_count": n,
                     "Status": "PREPARED" if n > 0 else "FAILED_NO_ATOMS"})
    else:
        out.unlink(missing_ok=True)
        rows.append({"Name": name, "PDBQT": str(out), "Atom_count": 0, "Status": "FAILED"})
pdbqt_df = pd.DataFrame(rows)
pdbqt_df = pdbqt_df[pdbqt_df["Status"].isin(["PREPARED", "REUSED"])].reset_index(drop=True)
pdbqt_df.to_csv(LIGAND_PDBQT_DIR / "ligand_pdbqt_preparation.csv", index=False)
print(f"Ligand PDBQT prepared: {len(pdbqt_df)}")
if pdbqt_df.empty:
    raise RuntimeError("No ligand PDBQT prepared")

Ligand PDBQT prepared: 6


In [6]:
# CELL 5 — Checkpointed docking (three seeds)
DOCKING_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = DOCKING_DIR / "docking_progress.csv"
CKPT_COLS = ["Name", "Seed", "Status", "Affinity_kcal_mol",
             "Pose_PDBQT", "Log", "Error"]

if CHECKPOINT.exists():
    ckpt = pd.read_csv(CHECKPOINT)
    for c in CKPT_COLS:
        if c not in ckpt.columns:
            ckpt[c] = ""
    ckpt = ckpt[CKPT_COLS]
else:
    ckpt = pd.DataFrame(columns=CKPT_COLS)
ckpt = ckpt.drop_duplicates(subset=["Name", "Seed"], keep="last").reset_index(drop=True)


def save_ckpt(df):
    tmp = CHECKPOINT.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(CHECKPOINT)


def parse_affinity(text):
    if not text:
        return np.nan
    m = re.search(r"REMARK\s+VINA RESULT:\s*(-?\d+(?:\.\d+)?)", text)
    if m:
        return float(m.group(1))
    for line in text.splitlines():
        m = re.match(r"^\s*1\s+(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)", line)
        if m:
            return float(m.group(1))
    return np.nan


def count_atoms(p):
    try:
        return sum(1 for L in Path(p).read_text(errors="replace").splitlines()
                   if L.startswith(("ATOM", "HETATM")))
    except Exception:
        return 0


def valid(row, expected_pose):
    if str(row.get("Status", "")).upper() not in ("COMPLETED", "REUSED"):
        return False
    try:
        aff = float(row.get("Affinity_kcal_mol"))
    except (TypeError, ValueError):
        return False
    if not np.isfinite(aff):
        return False
    p = Path(str(row.get("Pose_PDBQT") or expected_pose))
    return p.exists() and p.stat().st_size > 100 and count_atoms(p) > 0


jobs_run = jobs_reused = jobs_failed = 0

for _, lig in pdbqt_df.iterrows():
    name = str(lig["Name"])
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    ligand_pdbqt = Path(lig["PDBQT"])
    comp_dir = DOCKING_DIR / safe
    comp_dir.mkdir(parents=True, exist_ok=True)

    for seed in SEEDS:
        pose_file = comp_dir / f"{safe}_seed{seed}.pdbqt"
        log_file = comp_dir / f"{safe}_seed{seed}.log"

        existing = ckpt[(ckpt["Name"].astype(str) == name)
                        & (ckpt["Seed"].astype(str) == str(seed))]
        if not existing.empty and valid(existing.iloc[-1], pose_file):
            jobs_reused += 1
            continue

        if pose_file.exists():
            pose_file.unlink()

        cmd = [
            str(VINA),
            "--receptor", str(RECEPTOR_PDBQT),
            "--ligand", str(ligand_pdbqt),
            "--center_x", str(BOX["center_x"]),
            "--center_y", str(BOX["center_y"]),
            "--center_z", str(BOX["center_z"]),
            "--size_x", str(BOX["size_x"]),
            "--size_y", str(BOX["size_y"]),
            "--size_z", str(BOX["size_z"]),
            "--exhaustiveness", str(EXHAUSTIVENESS),
            "--num_modes", str(NUM_MODES),
            "--energy_range", str(ENERGY_RANGE),
            "--cpu", str(CPU),
            "--seed", str(seed),
            "--out", str(pose_file),
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        log_file.write_text(" ".join(cmd) + "\n\n"
                            + (r.stdout or "") + "\n" + (r.stderr or ""))

        if r.returncode != 0:
            rec = {"Name": name, "Seed": seed, "Status": "FAILED",
                   "Affinity_kcal_mol": np.nan,
                   "Pose_PDBQT": str(pose_file), "Log": str(log_file),
                   "Error": (r.stderr or r.stdout or "")[-1000:]}
            jobs_failed += 1
        else:
            aff = parse_affinity(r.stdout or "")
            n = count_atoms(pose_file)
            if n == 0 or not np.isfinite(aff):
                rec = {"Name": name, "Seed": seed, "Status": "FAILED_INVALID_POSE",
                       "Affinity_kcal_mol": np.nan,
                       "Pose_PDBQT": str(pose_file), "Log": str(log_file),
                       "Error": "invalid pose or affinity"}
                jobs_failed += 1
            else:
                rec = {"Name": name, "Seed": seed, "Status": "COMPLETED",
                       "Affinity_kcal_mol": aff,
                       "Pose_PDBQT": str(pose_file), "Log": str(log_file),
                       "Error": ""}
                jobs_run += 1

        ckpt = ckpt[~((ckpt["Name"].astype(str) == name)
                      & (ckpt["Seed"].astype(str) == str(seed)))].copy()
        ckpt = pd.concat([ckpt, pd.DataFrame([rec])], ignore_index=True)
        save_ckpt(ckpt)

print(f"Reused: {jobs_reused}  New: {jobs_run}  Failed: {jobs_failed}")
if jobs_failed:
    print(f"⚠ {jobs_failed} docking jobs failed — check {CHECKPOINT}")

Reused: 18  New: 0  Failed: 0


In [7]:

# CELL 5b — Control docking: 20 DDR1 actives vs 50 property-matched decoys
# Single Vina seed: 7
# Diagnostic benchmark; does not validate absolute docking-score thresholds.

import hashlib
import json
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem as _AllChem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import roc_auc_score, roc_curve

CONTROL_DIR = DOCKING_DIR / "controls"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_CHECKPOINT = CONTROL_DIR / "control_docking_progress.csv"

# ---------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------
CONTROL_ACTIVE_PCHEMBL = 7.0
CONTROL_N_ACTIVES = 20
CONTROL_N_DECOYS = 50
CONTROL_SEED = 7
CONTROL_SELECTION_SEED = 20260914
CONTROL_BOOTSTRAP_SEED = 20260914
CONTROL_N_BOOTSTRAP = 2000

CONTROL_TOL = {
    "heavy_atoms": 5,
    "mw": 75.0,
    "logp": 1.5,
    "hbd": 2,
    "hba": 2,
}

print("Control docking configuration:")
print(f"  Target actives: {CONTROL_N_ACTIVES}")
print(f"  Target decoys:  {CONTROL_N_DECOYS}")
print(f"  Vina seed:      {CONTROL_SEED}")

# ---------------------------------------------------------------
# Property helper
# ---------------------------------------------------------------
def _props(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return {
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "mw": float(Descriptors.MolWt(mol)),
        "logp": float(Descriptors.MolLogP(mol)),
        "hbd": int(rdMolDescriptors.CalcNumHBD(mol)),
        "hba": int(rdMolDescriptors.CalcNumHBA(mol)),
    }


def _murcko(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    return scaffold if scaffold else f"ACYCLIC::{smi}"


# ---------------------------------------------------------------
# 1. Select actives: pChEMBL >= 7, unique scaffolds, not in training
# ---------------------------------------------------------------
chembl = pd.read_excel(PATHS["DATA_RAW"] / "CHEMBL5319.xlsx")
chembl.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in chembl.columns
]

pcol = next((c for c in chembl.columns if "pchembl" in c), None)
smcol = next((c for c in chembl.columns if "smiles" in c), None)

if pcol is None or smcol is None:
    raise KeyError(
        f"ChEMBL file needs pChEMBL and SMILES columns; found {list(chembl.columns)}"
    )

chembl["_p"] = pd.to_numeric(chembl[pcol], errors="coerce")
chembl["_canon"] = chembl[smcol].apply(strip_salts_and_neutralize)

# All available DDR1 ChEMBL structures are excluded from the decoy pool.
_all_known_chembl = set(chembl["_canon"].dropna())

active_pool = chembl[
    (chembl["_p"] >= CONTROL_ACTIVE_PCHEMBL)
    & chembl["_canon"].notna()
].copy()

active_pool = active_pool.drop_duplicates("_canon")

train_path = (
    PATHS["DATA_PROCESSED"] / "training_scaffold_membership.csv"
)
if not train_path.exists():
    raise FileNotFoundError(
        f"{train_path} not found. Run the model notebook first."
    )

trained = set(
    pd.read_csv(train_path)["canonical_smiles"].astype(str)
)
n_before = len(active_pool)
active_pool = active_pool[
    ~active_pool["_canon"].isin(trained)
].copy()

print(
    f"Removed {n_before - len(active_pool)} training compounds "
    "from active pool."
)

active_pool["_scaffold"] = active_pool["_canon"].apply(_murcko)
active_pool = active_pool[
    active_pool["_scaffold"].notna()
].copy()

# Retain the highest measured pChEMBL per distinct scaffold.
active_pool = (
    active_pool.sort_values("_p", ascending=False)
    .drop_duplicates("_scaffold")
)

if len(active_pool) < CONTROL_N_ACTIVES:
    raise RuntimeError(
        f"Only {len(active_pool)} distinct-scaffold actives are available; "
        f"{CONTROL_N_ACTIVES} are required. Do not silently reduce the target."
    )

actives = (
    active_pool.sample(
        n=CONTROL_N_ACTIVES,
        random_state=CONTROL_SELECTION_SEED
    )[["_canon", "_p", "_scaffold"]]
    .rename(columns={
        "_canon": "canon_smiles",
        "_p": "pChEMBL",
        "_scaffold": "scaffold"
    })
    .reset_index(drop=True)
)

print(f"Selected actives: {len(actives)}")
print(f"Active pChEMBL range: {actives['pChEMBL'].min():.2f}–"
      f"{actives['pChEMBL'].max():.2f}")

# ---------------------------------------------------------------
# 2. Build a reproducible, property-matched decoy pool
# ---------------------------------------------------------------
drugbank = pd.read_excel(
    PATHS["DATA_RAW"] / "FDA_DrugBank_cleaned.xlsx"
)

dbsmi = next(
    (c for c in ("Cleaned_SMILES", "SMILES", "smiles")
     if c in drugbank.columns),
    None
)
if dbsmi is None:
    raise KeyError(
        f"No DrugBank SMILES column found: {list(drugbank.columns)}"
    )

drugbank["_canon"] = drugbank[dbsmi].apply(strip_salts_and_neutralize)
drugbank = drugbank[
    drugbank["_canon"].notna()
].drop_duplicates("_canon").copy()

# Exclude all known DDR1 ChEMBL compounds, selected actives,
# and current docking candidates.
candidate_smiles = set()
try:
    candidate_smiles = set(
        prepared_df["Canonical_SMILES"].dropna().astype(str)
    )
except (NameError, KeyError, AttributeError):
    pass

exclude_smiles = (
    _all_known_chembl
    | set(actives["canon_smiles"])
    | candidate_smiles
)

n_db_before = len(drugbank)
drugbank = drugbank[
    ~drugbank["_canon"].isin(exclude_smiles)
].copy()

print(
    f"Removed {n_db_before - len(drugbank)} known DDR1, active, "
    "or candidate structures from DrugBank."
)

# Calculate mean physicochemical properties of the selected actives.
active_props = pd.DataFrame([
    _props(smi) for smi in actives["canon_smiles"]
]).dropna()

if len(active_props) != len(actives):
    raise RuntimeError("Some active structures failed property calculation.")

active_means = active_props.mean(numeric_only=True)

# Randomize the candidate order reproducibly before matching.
drugbank = drugbank.sample(
    frac=1,
    random_state=CONTROL_SELECTION_SEED
).reset_index(drop=True)

# Compute properties for all available decoy candidates.
drugbank["_props"] = drugbank["_canon"].apply(_props)
drugbank = drugbank[
    drugbank["_props"].notna()
].copy()

for prop in CONTROL_TOL:
    drugbank[prop] = drugbank["_props"].apply(lambda x: x[prop])

def _within_tolerance(row):
    return all(
        abs(float(row[prop]) - float(active_means[prop])) <= tol
        for prop, tol in CONTROL_TOL.items()
    )

matched = drugbank[
    drugbank.apply(_within_tolerance, axis=1)
].copy()

if len(matched) < CONTROL_N_DECOYS:
    raise RuntimeError(
        f"Only {len(matched)} property-matched decoys are available; "
        f"{CONTROL_N_DECOYS} are required. Consider a prespecified broader "
        "decoy source or matching strategy; do not silently reduce the target."
    )

decoys = (
    matched.head(CONTROL_N_DECOYS)[["_canon"]]
    .rename(columns={"_canon": "canon_smiles"})
    .reset_index(drop=True)
)

print(f"Selected property-matched decoys: {len(decoys)}")

# Save the exact selected sets for audit and reproducibility.
actives.to_csv(CONTROL_DIR / "selected_active_set.csv", index=False)
decoys.to_csv(CONTROL_DIR / "selected_decoy_set.csv", index=False)

# ---------------------------------------------------------------
# 3. Checkpointed docking — exactly seed 7
# ---------------------------------------------------------------
CKPT_COLS = [
    "label", "smiles", "seed", "status", "affinity", "error"
]

if CONTROL_CHECKPOINT.exists():
    ctrl_ckpt = pd.read_csv(CONTROL_CHECKPOINT)
    for col in CKPT_COLS:
        if col not in ctrl_ckpt.columns:
            ctrl_ckpt[col] = np.nan
    ctrl_ckpt = ctrl_ckpt[CKPT_COLS]
else:
    ctrl_ckpt = pd.DataFrame(columns=CKPT_COLS)

ctrl_ckpt = ctrl_ckpt.drop_duplicates(
    subset=["label", "smiles", "seed"],
    keep="last"
).reset_index(drop=True)


def _save_ctrl_ckpt(df):
    tmp = CONTROL_CHECKPOINT.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(CONTROL_CHECKPOINT)


def _prepare_and_dock(smi, label, seed):
    """Prepare one ligand and dock with the specified Vina seed."""
    tag = hashlib.sha256(smi.encode("utf-8")).hexdigest()[:12]

    sdf = CONTROL_DIR / f"{label}_{tag}.sdf"
    pdbqt = CONTROL_DIR / f"{label}_{tag}.pdbqt"
    pose = CONTROL_DIR / f"{label}_{tag}_seed{seed}_docked.pdbqt"

    if not pdbqt.exists() or pdbqt.stat().st_size < 50:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None, "SMILES parse failed"

        mol = Chem.AddHs(mol)
        embed_status = _AllChem.EmbedMolecule(
            mol,
            randomSeed=CONTROL_SELECTION_SEED,
            useRandomCoords=True
        )
        if embed_status != 0:
            return None, "3D embedding failed"

        try:
            if _AllChem.MMFFHasAllMoleculeParams(mol):
                _AllChem.MMFFOptimizeMolecule(
                    mol,
                    mmffVariant="MMFF94",
                    maxIters=500
                )
        except Exception as exc:
            return None, f"MMFF optimization failed: {exc}"

        writer = Chem.SDWriter(str(sdf))
        writer.write(mol)
        writer.close()

        meeko_result = subprocess.run(
            [
                str(MEEKO_LIGAND),
                "-i", str(sdf),
                "-o", str(pdbqt)
            ],
            capture_output=True,
            text=True
        )

        if meeko_result.returncode != 0 or not pdbqt.exists():
            return None, (
                f"Meeko failed: {meeko_result.stderr[-300:]}"
            )

    command = [
        str(VINA),
        "--receptor", str(RECEPTOR_PDBQT),
        "--ligand", str(pdbqt),
        "--center_x", str(BOX["center_x"]),
        "--center_y", str(BOX["center_y"]),
        "--center_z", str(BOX["center_z"]),
        "--size_x", str(BOX["size_x"]),
        "--size_y", str(BOX["size_y"]),
        "--size_z", str(BOX["size_z"]),
        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--num_modes", str(NUM_MODES),
        "--energy_range", str(ENERGY_RANGE),
        "--cpu", str(CPU),
        "--seed", str(seed),
        "--out", str(pose)
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0 or not pose.exists():
        return None, f"Vina failed: {result.stderr[-300:]}"

    affinity = parse_affinity(result.stdout or "")
    if not np.isfinite(affinity):
        return None, "Could not parse Vina affinity"

    return float(affinity), ""


for label, frame in (("active", actives), ("decoy", decoys)):
    for smi in frame["canon_smiles"]:
        existing = ctrl_ckpt[
            (ctrl_ckpt["label"] == label)
            & (ctrl_ckpt["smiles"] == smi)
            & (pd.to_numeric(ctrl_ckpt["seed"], errors="coerce")
               == CONTROL_SEED)
            & (ctrl_ckpt["status"] == "ok")
        ]

        if not existing.empty:
            continue

        affinity, error = _prepare_and_dock(
            smi, label, CONTROL_SEED
        )

        record = {
            "label": label,
            "smiles": smi,
            "seed": CONTROL_SEED,
            "status": "ok" if affinity is not None else "failed",
            "affinity": affinity if affinity is not None else np.nan,
            "error": error
        }

        ctrl_ckpt = ctrl_ckpt[
            ~(
                (ctrl_ckpt["label"] == label)
                & (ctrl_ckpt["smiles"] == smi)
                & (pd.to_numeric(ctrl_ckpt["seed"], errors="coerce")
                   == CONTROL_SEED)
            )
        ].copy()

        ctrl_ckpt = pd.concat(
            [ctrl_ckpt, pd.DataFrame([record])],
            ignore_index=True
        )
        _save_ctrl_ckpt(ctrl_ckpt)

# ---------------------------------------------------------------
# 4. Restrict results to the currently selected compounds
# ---------------------------------------------------------------
selected = {
    ("active", smi) for smi in actives["canon_smiles"]
} | {
    ("decoy", smi) for smi in decoys["canon_smiles"]
}

current = ctrl_ckpt[
    ctrl_ckpt.apply(
        lambda r: (r["label"], r["smiles"]) in selected
        and pd.to_numeric(r["seed"], errors="coerce") == CONTROL_SEED,
        axis=1
    )
].copy()

current.to_csv(
    CONTROL_DIR / "control_docking_all_results.csv",
    index=False
)

successful = current[
    (current["status"] == "ok")
    & pd.to_numeric(current["affinity"], errors="coerce").notna()
].copy()

successful["affinity"] = pd.to_numeric(
    successful["affinity"], errors="coerce"
)

n_active_ok = int((successful["label"] == "active").sum())
n_decoy_ok = int((successful["label"] == "decoy").sum())

print("\nDocking completion:")
print(f"  Active successes: {n_active_ok}/{CONTROL_N_ACTIVES}")
print(f"  Decoy successes:  {n_decoy_ok}/{CONTROL_N_DECOYS}")
print(f"  Vina seed:        {CONTROL_SEED}")

successful.to_csv(
    CONTROL_DIR / "control_docking_results.csv",
    index=False
)

# ---------------------------------------------------------------
# 5. ROC analysis — only if all 20 + 50 compounds succeeded
# ---------------------------------------------------------------
full_success = (
    n_active_ok == CONTROL_N_ACTIVES
    and n_decoy_ok == CONTROL_N_DECOYS
)

summary = {
    "target_n_actives": CONTROL_N_ACTIVES,
    "target_n_decoys": CONTROL_N_DECOYS,
    "successful_n_actives": n_active_ok,
    "successful_n_decoys": n_decoy_ok,
    "vina_seed": CONTROL_SEED,
    "selection_seed": CONTROL_SELECTION_SEED,
    "full_target_set_successfully_docked": bool(full_success),
    "auc": None,
    "auc_ci95": None,
    "ef_top_5_percent": None,
    "interpretation": (
        "Benchmark not calculated because one or more selected compounds "
        "failed docking. Resolve failures and rerun this cell."
    )
}

if full_success:
    y = (successful["label"] == "active").astype(int).to_numpy()

    # Lower (more negative) Vina affinity is treated as a stronger score.
    scores = -successful["affinity"].to_numpy(dtype=float)

    auc = float(roc_auc_score(y, scores))
    fpr, tpr, _ = roc_curve(y, scores)

    # Stratified bootstrap: resample actives and decoys independently.
    active_scores = scores[y == 1]
    decoy_scores = scores[y == 0]
    rng = np.random.default_rng(CONTROL_BOOTSTRAP_SEED)

    bootstrap_auc = []
    for _ in range(CONTROL_N_BOOTSTRAP):
        a = rng.choice(
            active_scores, size=len(active_scores), replace=True
        )
        d = rng.choice(
            decoy_scores, size=len(decoy_scores), replace=True
        )

        y_boot = np.concatenate([
            np.ones(len(a), dtype=int),
            np.zeros(len(d), dtype=int)
        ])
        score_boot = np.concatenate([a, d])
        bootstrap_auc.append(roc_auc_score(y_boot, score_boot))

    auc_low, auc_high = np.percentile(
        bootstrap_auc, [2.5, 97.5]
    ).astype(float)

    # EF at top 5%; ties/order may affect a small top fraction.
    order = np.argsort(-scores)
    ranked_labels = y[order]
    top_n = max(1, int(np.ceil(0.05 * len(y))))
    observed_rate = ranked_labels[:top_n].mean()
    overall_rate = y.mean()
    ef5 = float(observed_rate / overall_rate)

    summary.update({
        "auc": auc,
        "auc_ci95": [float(auc_low), float(auc_high)],
        "ef_top_5_percent": ef5,
        "interpretation": (
            "Single-seed control benchmark using 20 experimentally defined "
            "DDR1 actives and 50 property-matched DrugBank decoys. "
            "The decoys are not experimentally confirmed inactive. "
            "This benchmark does not validate absolute docking-score tiers."
        )
    })

    plt.figure(figsize=(6, 6))
    plt.plot(
        fpr, tpr,
        label=f"AUC = {auc:.3f} (95% bootstrap CI {auc_low:.3f}–{auc_high:.3f})"
    )
    plt.plot([0, 1], [0, 1], "k--", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("DDR1 control docking: 20 actives vs 50 decoys (seed 7)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(
        CONTROL_DIR / "control_roc.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(f"\nROC-AUC: {auc:.3f}")
    print(f"95% stratified bootstrap CI: {auc_low:.3f}–{auc_high:.3f}")
    print(f"EF@5%: {ef5:.3f}")

with open(CONTROL_DIR / "control_roc_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

print(f"\nSaved control results to: {CONTROL_DIR}")
print(f"Summary: {CONTROL_DIR / 'control_roc_summary.json'}")
print(
    "\nInterpretation caution: one seed reduces computational cost but "
    "does not measure pose/score variability across seeds. DrugBank decoys "
    "are presumed decoys, not experimentally verified DDR1-inactive compounds."
)


Control docking configuration:
  Target actives: 20
  Target decoys:  50
  Vina seed:      7
Removed 756 training compounds from active pool.
Selected actives: 20
Active pChEMBL range: 7.07–8.17
Removed 11 known DDR1, active, or candidate structures from DrugBank.
Selected property-matched decoys: 50

Docking completion:
  Active successes: 20/20
  Decoy successes:  50/50
  Vina seed:        7

ROC-AUC: 0.869
95% stratified bootstrap CI: 0.776–0.941
EF@5%: 3.500

Saved control results to: /content/drive/MyDrive/DDR1_Workflow/candidate_docking/MANUSCRIPT_GRADE_DDR1_ANALYSIS/candidate_docking/controls
Summary: /content/drive/MyDrive/DDR1_Workflow/candidate_docking/MANUSCRIPT_GRADE_DDR1_ANALYSIS/candidate_docking/controls/control_roc_summary.json

Interpretation caution: one seed reduces computational cost but does not measure pose/score variability across seeds. DrugBank decoys are presumed decoys, not experimentally verified DDR1-inactive compounds.


In [8]:
# CELL 6 — Pose reproducibility with graph-validated, symmetry-corrected DIRECT RMSD
# RMSD is calculated in the receptor coordinate frame; NO superposition is applied.
# Atom correspondence is derived from Meeko's REMARK SMILES / REMARK SMILES IDX metadata.
# If atom mapping or molecular-graph validation fails, the result remains UNRESOLVED.

POSE_REPRO_DIR.mkdir(parents=True, exist_ok=True)

_AD_TYPE_TO_ELEMENT = {
    "C": "C", "A": "C",
    "N": "N", "NA": "N", "NS": "N",
    "O": "O", "OA": "O", "OS": "O",
    "S": "S", "SA": "S", "P": "P",
    "F": "F", "CL": "CL", "BR": "BR", "I": "I",
    "H": "H", "HD": "H", "HS": "H",
}


# ---------------------------------------------------------------------
# 1. Read MODEL 1 and the corresponding Meeko atom-index metadata
# ---------------------------------------------------------------------
def _read_first_pdbqt_model(pdbqt_path):
    """
    Read MODEL 1 only when the file contains multiple models.
    Preserve metadata preceding MODEL 1 and metadata within MODEL 1.

    Meeko REMARK SMILES IDX pairs are interpreted as:
        1-based SMILES atom index, PDBQT atom serial.

    Identical metadata pairs may be repeated across the header/MODEL 1.
    Repeated identical pairs are deduplicated; conflicting mappings fail.
    """
    path = Path(pdbqt_path)

    if not path.exists():
        raise FileNotFoundError(f"PDBQT file not found: {path}")

    lines = path.read_text(errors="replace").splitlines()
    has_models = any(line.startswith("MODEL") for line in lines)

    selected = []

    if has_models:
        model_started = False

        for line in lines:
            if line.startswith("MODEL"):
                if model_started:
                    break
                model_started = True
                selected.append(line)
                continue

            selected.append(line)

            if model_started and line.startswith("ENDMDL"):
                break

        if not model_started:
            raise ValueError(f"Could not locate MODEL 1 in {path}")
    else:
        selected = lines

    smiles = None
    idx_pairs = []
    records = []
    in_model = not has_models

    for line in selected:
        if line.startswith("MODEL"):
            in_model = True
            continue

        if line.startswith("ENDMDL"):
            in_model = False
            continue

        # SMILES metadata can appear in the file header or MODEL 1.
        if line.startswith("REMARK SMILES ") and not line.startswith(
            "REMARK SMILES IDX"
        ):
            value = line[len("REMARK SMILES "):].strip()

            if value and smiles is None:
                smiles = value.split()[0]

        if line.startswith("REMARK SMILES IDX"):
            values = line[len("REMARK SMILES IDX"):].split()

            if len(values) % 2:
                raise ValueError(
                    f"Malformed REMARK SMILES IDX record in {path}: {line}"
                )

            for j in range(0, len(values), 2):
                # Correct Meeko convention verified against the prepared
                # Cabozantinib PDBQT:
                # first = 1-based SMILES atom index; second = PDBQT serial.
                smiles_idx_1based = int(values[j])
                serial = int(values[j + 1])

                if smiles_idx_1based < 1:
                    raise ValueError(
                        f"Invalid SMILES atom index in {path}: "
                        f"{smiles_idx_1based}"
                    )

                idx_pairs.append((serial, smiles_idx_1based - 1))

        if not line.startswith(("ATOM", "HETATM")) or not in_model:
            continue

        try:
            serial = int(line[6:11])
            atom_name = line[12:16].strip()
            xyz = np.array(
                [
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54]),
                ],
                dtype=float,
            )
        except (ValueError, IndexError) as exc:
            raise ValueError(
                f"Could not parse atom record in {path}: {line}"
            ) from exc

        atom_type = line[77:79].strip().upper() if len(line) >= 79 else ""
        element = _AD_TYPE_TO_ELEMENT.get(atom_type, atom_type)

        if not element:
            raise ValueError(
                f"Missing or unknown AutoDock atom type for atom {serial} in {path}"
            )

        records.append({
            "serial": serial,
            "name": atom_name,
            "atype": atom_type,
            "element": element,
            "xyz": xyz,
        })

    if not records:
        raise ValueError(f"No atom records found in MODEL 1: {path}")

    if not smiles:
        raise ValueError(
            f"Missing Meeko REMARK SMILES metadata in {path}"
        )

    if not idx_pairs:
        raise ValueError(
            f"Missing Meeko REMARK SMILES IDX metadata in {path}"
        )

    serial_to_smiles_idx = {}
    smiles_idx_to_serial = {}

    # Metadata may be repeated verbatim; deduplicate exact pairs, but reject
    # any conflicting serial-to-atom or atom-to-serial assignments.
    for serial, smiles_idx in sorted(set(idx_pairs)):
        if serial in serial_to_smiles_idx and serial_to_smiles_idx[serial] != smiles_idx:
            raise ValueError(
                f"Conflicting SMILES IDX assignments for PDBQT serial {serial}: {path}"
            )

        if smiles_idx in smiles_idx_to_serial and smiles_idx_to_serial[smiles_idx] != serial:
            raise ValueError(
                f"Conflicting PDBQT serial assignments for SMILES atom "
                f"{smiles_idx + 1}: {path}"
            )

        serial_to_smiles_idx[serial] = smiles_idx
        smiles_idx_to_serial[smiles_idx] = serial

    if len(serial_to_smiles_idx) != len(smiles_idx_to_serial):
        raise ValueError(f"SMILES IDX metadata is not bijective: {path}")

    return records, smiles, serial_to_smiles_idx


# ---------------------------------------------------------------------
# 2. Load the prepared SDF reference graph
# ---------------------------------------------------------------------
def _reference_heavy_graph(sdf_path):
    """Return the prepared ligand's heavy-atom graph in SDF atom order."""
    mol = Chem.MolFromMolFile(
        str(sdf_path),
        removeHs=False,
        sanitize=True,
    )

    if mol is None:
        raise ValueError(f"RDKit could not read ligand SDF: {sdf_path}")

    heavy_mol = Chem.RemoveHs(mol)

    if heavy_mol is None or heavy_mol.GetNumAtoms() == 0:
        raise ValueError(f"No heavy atoms found in reference SDF: {sdf_path}")

    return heavy_mol


# ---------------------------------------------------------------------
# 3. Validate graph correspondence and map pose coordinates to SDF order
# ---------------------------------------------------------------------
def _graph_mapping_to_reference(reference_mol, pose_smiles_mol, source):
    """
    Return a tuple mapping:
        pose-SMILES atom index -> reference-SDF heavy-atom index.

    Requires matching atom and bond counts and a stereochemistry-aware
    full-graph mapping. No coordinate-based atom assignment is used.
    """
    if pose_smiles_mol is None:
        raise ValueError(f"RDKit could not parse Meeko SMILES: {source}")

    if pose_smiles_mol.GetNumAtoms() != reference_mol.GetNumAtoms():
        raise ValueError(
            f"Heavy-atom count mismatch for {source}: "
            f"Meeko SMILES={pose_smiles_mol.GetNumAtoms()}, "
            f"SDF={reference_mol.GetNumAtoms()}"
        )

    if pose_smiles_mol.GetNumBonds() != reference_mol.GetNumBonds():
        raise ValueError(
            f"Bond-count mismatch for {source}: "
            f"Meeko SMILES={pose_smiles_mol.GetNumBonds()}, "
            f"SDF={reference_mol.GetNumBonds()}"
        )

    matches = reference_mol.GetSubstructMatches(
        pose_smiles_mol,
        uniquify=False,
        useChirality=True,
        maxMatches=10000,
    )

    valid_matches = []

    for match in matches:
        if len(match) != reference_mol.GetNumAtoms():
            continue

        if len(set(match)) != reference_mol.GetNumAtoms():
            continue

        elements_match = all(
            pose_smiles_mol.GetAtomWithIdx(i).GetAtomicNum()
            == reference_mol.GetAtomWithIdx(match[i]).GetAtomicNum()
            for i in range(pose_smiles_mol.GetNumAtoms())
        )

        if elements_match:
            valid_matches.append(tuple(int(x) for x in match))

    if not valid_matches:
        raise ValueError(
            "No stereochemistry-aware full-graph mapping between "
            f"Meeko SMILES and reference SDF: {source}"
        )

    # A deterministic valid mapping is sufficient because RMSD is subsequently
    # minimized over graph-valid automorphisms of the reference ligand.
    return sorted(set(valid_matches))[0]


def _pose_coordinates_in_reference_order(pdbqt_path, reference_mol):
    """
    Map heavy-atom coordinates using:
      PDBQT serial -> Meeko SMILES atom index -> reference SDF atom index.
    """
    records, smiles, serial_to_smiles_idx = _read_first_pdbqt_model(
        pdbqt_path
    )

    smiles_mol = Chem.MolFromSmiles(smiles)

    if smiles_mol is None:
        raise ValueError(
            f"Could not parse REMARK SMILES in {pdbqt_path}"
        )

    pose_to_reference = _graph_mapping_to_reference(
        reference_mol,
        smiles_mol,
        str(pdbqt_path),
    )

    n_heavy = reference_mol.GetNumAtoms()
    coordinates_by_reference_idx = {}

    heavy_serials = {
        atom["serial"] for atom in records
        if atom["element"].upper() not in {"H", "D"}
    }
    metadata_serials = set(serial_to_smiles_idx)

    if metadata_serials != heavy_serials:
        missing = sorted(heavy_serials - metadata_serials)
        extra = sorted(metadata_serials - heavy_serials)
        raise ValueError(
            f"SMILES IDX metadata does not exactly match MODEL 1 heavy atoms "
            f"in {pdbqt_path}; missing={missing}, extra={extra}"
        )

    for atom in records:
        # Hydrogens are excluded from the RMSD calculation.
        if atom["element"] == "H":
            continue

        serial = atom["serial"]

        if serial not in serial_to_smiles_idx:
            raise ValueError(
                f"Heavy-atom serial {serial} has no SMILES IDX mapping: "
                f"{pdbqt_path}"
            )

        smiles_idx = serial_to_smiles_idx[serial]

        if not 0 <= smiles_idx < smiles_mol.GetNumAtoms():
            raise ValueError(
                f"Heavy atom {serial} maps to invalid SMILES index "
                f"{smiles_idx}: {pdbqt_path}"
            )

        reference_idx = pose_to_reference[smiles_idx]

        if reference_idx in coordinates_by_reference_idx:
            raise ValueError(
                f"Multiple heavy atoms map to reference atom {reference_idx}: "
                f"{pdbqt_path}"
            )

        expected_element = (
            reference_mol.GetAtomWithIdx(reference_idx)
            .GetSymbol()
            .upper()
        )

        if atom["element"].upper() != expected_element:
            raise ValueError(
                f"Element mismatch for atom serial {serial}: "
                f"PDBQT={atom['element']}, reference={expected_element}"
            )

        coordinates_by_reference_idx[reference_idx] = atom["xyz"]

    expected_indices = set(range(n_heavy))
    observed_indices = set(coordinates_by_reference_idx)

    if observed_indices != expected_indices:
        missing = sorted(expected_indices - observed_indices)
        extra = sorted(observed_indices - expected_indices)

        raise ValueError(
            f"Incomplete heavy-atom mapping in {pdbqt_path}; "
            f"missing={missing}, extra={extra}"
        )

    coords = np.asarray(
        [coordinates_by_reference_idx[i] for i in range(n_heavy)],
        dtype=float,
    )

    if coords.shape != (n_heavy, 3) or not np.isfinite(coords).all():
        raise ValueError(
            f"Invalid heavy-atom coordinates in {pdbqt_path}"
        )

    return coords


# ---------------------------------------------------------------------
# 4. Enumerate graph-valid symmetry mappings and calculate direct RMSD
# ---------------------------------------------------------------------
def _heavy_graph_automorphisms(reference_mol):
    """Return graph-valid heavy-atom permutations in reference atom order."""
    matches = reference_mol.GetSubstructMatches(
        reference_mol,
        uniquify=False,
        useChirality=True,
        maxMatches=10000,
    )

    permutations = sorted({
        tuple(int(x) for x in match)
        for match in matches
        if len(match) == reference_mol.GetNumAtoms()
        and len(set(match)) == reference_mol.GetNumAtoms()
    })

    identity = tuple(range(reference_mol.GetNumAtoms()))

    if identity not in permutations:
        permutations.insert(0, identity)

    if not permutations:
        raise ValueError("No graph-valid heavy-atom automorphisms found")

    return permutations


def _symmetry_corrected_direct_rmsd(coords_a, coords_b, automorphisms):
    """
    Calculate minimum heavy-atom RMSD over graph-valid atom permutations.
    Coordinates remain in the receptor frame; no superposition is applied.
    """
    if (
        coords_a.shape != coords_b.shape
        or coords_a.ndim != 2
        or coords_a.shape[1] != 3
    ):
        raise ValueError("Coordinate arrays must have identical (N, 3) shapes")

    values = []

    for permutation in automorphisms:
        permuted_b = coords_b[np.asarray(permutation, dtype=int)]
        rmsd = np.sqrt(np.mean(np.sum((coords_a - permuted_b) ** 2, axis=1)))
        values.append(float(rmsd))

    if not values:
        raise ValueError("No valid atom permutations available for RMSD")

    return min(values)


# ---------------------------------------------------------------------
# 5. Collect candidates with successful docking for every configured seed
# ---------------------------------------------------------------------
successful = ckpt[
    ckpt["Status"].astype(str).str.upper().isin(["COMPLETED", "REUSED"])
].copy()

by_cand = {}

for _, row in successful.iterrows():
    try:
        seed = int(row["Seed"])
        pose_value = row["Pose_PDBQT"]

        if pd.isna(pose_value) or not str(pose_value).strip():
            continue

        pose_path = Path(str(pose_value))

    except (TypeError, ValueError):
        continue

    if pose_path.exists() and pose_path.stat().st_size > 100:
        by_cand.setdefault(str(row["Name"]), {})[seed] = pose_path

complete = {
    name: files
    for name, files in by_cand.items()
    if all(seed in files for seed in SEEDS)
}

print(
    f"Candidates with all {len(SEEDS)} seeds and existing pose files: "
    f"{len(complete)}"
)


# Resolve each candidate to its prepared SDF from Cell 3.
_name_to_sdf = {}

for _, prep_row in prepared_df.iterrows():
    name = str(prep_row["Name"])
    sdf_value = prep_row.get("SDF")

    if pd.notna(sdf_value) and str(sdf_value).strip():
        _name_to_sdf[name] = Path(str(sdf_value))


# ---------------------------------------------------------------------
# 6. Calculate all three pairwise RMSDs and save the existing CSV output
# ---------------------------------------------------------------------
pair_order = [(7, 42), (7, 2026), (42, 2026)]
rows = []

for name, files in sorted(complete.items()):

    result = {
        "Name": name,
        "Seed7_vs_42_RMSD_A": np.nan,
        "Seed7_vs_2026_RMSD_A": np.nan,
        "Seed42_vs_2026_RMSD_A": np.nan,
        "Max_pairwise_RMSD_A": np.nan,
        "Mean_pairwise_RMSD_A": np.nan,
        # Retained for downstream compatibility; this stores the maximum
        # Kabsch/superposition diagnostic, not the consistency-gate RMSD.
        "Internal_conformer_RMSD_max_A": np.nan,
        "Mapped_atoms_min": np.nan,
        "Graph_automorphisms_considered": np.nan,
        "Pose_consistency": "UNRESOLVED",
        "QC_note": "",
    }

    try:
        if name not in _name_to_sdf:
            raise FileNotFoundError(
                f"No prepared SDF path found for candidate {name}"
            )

        reference_mol = _reference_heavy_graph(_name_to_sdf[name])
        automorphisms = _heavy_graph_automorphisms(reference_mol)

        coords = {
            seed: _pose_coordinates_in_reference_order(
                files[seed], reference_mol
            )
            for seed in SEEDS
        }

        expected_shape = (reference_mol.GetNumAtoms(), 3)

        for seed in SEEDS:
            if coords[seed].shape != expected_shape:
                raise ValueError(
                    f"Seed {seed} coordinate shape {coords[seed].shape}; "
                    f"expected {expected_shape}"
                )

        direct = {
            pair: _symmetry_corrected_direct_rmsd(
                coords[pair[0]],
                coords[pair[1]],
                automorphisms,
            )
            for pair in pair_order
        }

        # Diagnostic only. These superposition-based values do NOT determine
        # Pose_consistency. The existing output column is retained for compatibility.
        superposition_diag = {
            pair: kabsch_rmsd(coords[pair[0]], coords[pair[1]])
            for pair in pair_order
        }

        max_direct = max(direct.values())

        result.update({
            "Seed7_vs_42_RMSD_A": direct[(7, 42)],
            "Seed7_vs_2026_RMSD_A": direct[(7, 2026)],
            "Seed42_vs_2026_RMSD_A": direct[(42, 2026)],
            "Max_pairwise_RMSD_A": max_direct,
            "Mean_pairwise_RMSD_A": float(np.mean(list(direct.values()))),
            "Internal_conformer_RMSD_max_A": max(superposition_diag.values()),
            "Mapped_atoms_min": reference_mol.GetNumAtoms(),
            "Graph_automorphisms_considered": len(automorphisms),
            "Pose_consistency": (
                "YES"
                if max_direct <= float(POSE_CONSISTENCY_THRESHOLD)
                else "NO"
            ),
            "QC_note": (
                "Validated Meeko SMILES IDX mapping; stereochemistry-aware "
                "full-graph mapping; symmetry-corrected direct RMSD; "
                "no superposition"
            ),
        })

    except Exception as exc:
        result["QC_note"] = (
            f"UNRESOLVED: {type(exc).__name__}: {exc}"
        )

    rows.append(result)


pose_repro_df = pd.DataFrame(rows)

output_csv = (
    POSE_REPRO_DIR / "three_seed_pose_reproducibility.csv"
)
pose_repro_df.to_csv(output_csv, index=False)

if not pose_repro_df.empty:
    print(
        pose_repro_df[
            [
                "Name",
                "Seed7_vs_42_RMSD_A",
                "Seed7_vs_2026_RMSD_A",
                "Seed42_vs_2026_RMSD_A",
                "Max_pairwise_RMSD_A",
                "Pose_consistency",
                "QC_note",
            ]
        ].to_string(index=False)
    )
else:
    print(
        "No candidate had successful docking results for all configured seeds."
    )

print(f"\nSaved: {output_csv}")
print(
    "Interpretation: pose consistency across seeds is a reproducibility "
    "measure, not proof of docking-score accuracy or biological activity."
)

Candidates with all 3 seeds and existing pose files: 6
         Name  Seed7_vs_42_RMSD_A  Seed7_vs_2026_RMSD_A  Seed42_vs_2026_RMSD_A  Max_pairwise_RMSD_A Pose_consistency                                                                                                                        QC_note
 Cabozantinib            0.061579              0.063698               0.065586             0.065586              YES Validated Meeko SMILES IDX mapping; stereochemistry-aware full-graph mapping; symmetry-corrected direct RMSD; no superposition
   Lenvatinib            0.091927              0.044198               0.075389             0.091927              YES Validated Meeko SMILES IDX mapping; stereochemistry-aware full-graph mapping; symmetry-corrected direct RMSD; no superposition
  Quizartinib            0.107406              0.169277               0.082857             0.169277              YES Validated Meeko SMILES IDX mapping; stereochemistry-aware full-graph mapping; symmetry-corrected

In [9]:
# CELL 7 — Representative pose selection
REPRESENTATIVE_DIR.mkdir(parents=True, exist_ok=True)
rep_rows = []
for _, row in pose_repro_df.iterrows():
    name = str(row["Name"])
    rmsds = {k: pd.to_numeric(row.get(k), errors="coerce")
             for k in ("Seed7_vs_42_RMSD_A", "Seed7_vs_2026_RMSD_A", "Seed42_vs_2026_RMSD_A")}
    if any(pd.isna(v) for v in rmsds.values()):
        rep_rows.append({"Name": name, "Representative_status": "UNRESOLVED"}); continue
    m7 = (rmsds["Seed7_vs_42_RMSD_A"] + rmsds["Seed7_vs_2026_RMSD_A"]) / 2
    m42 = (rmsds["Seed7_vs_42_RMSD_A"] + rmsds["Seed42_vs_2026_RMSD_A"]) / 2
    m2026 = (rmsds["Seed7_vs_2026_RMSD_A"] + rmsds["Seed42_vs_2026_RMSD_A"]) / 2
    cent = {7: m7, 42: m42, 2026: m2026}
    rep_seed = sorted(cent.items(), key=lambda kv: (kv[1], kv[0]))[0][0]
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    dest = REPRESENTATIVE_DIR / f"{safe}_representative.pdbqt"
    _sh.copy2(by_cand[name][rep_seed], dest)
    rep_rows.append({"Name": name, "Representative_seed": rep_seed,
                     "Representative_mean_pairwise_RMSD_A": cent[rep_seed],
                     "Max_pairwise_RMSD_A": row.get("Max_pairwise_RMSD_A"),
                     "Pose_consistency": row.get("Pose_consistency"),
                     "Representative_pose_PDBQT": str(dest),
                     "Representative_status": "SELECTED"})
representative_df = pd.DataFrame(rep_rows)
representative_df.to_csv(REPRESENTATIVE_DIR / "representative_pose_selection.csv", index=False)
print(representative_df[["Name", "Representative_seed", "Representative_status"]].to_string(index=False))

         Name  Representative_seed Representative_status
 Cabozantinib                    7              SELECTED
   Lenvatinib                 2026              SELECTED
  Quizartinib                   42              SELECTED
Selpercatinib                    7              SELECTED
    Sorafenib                    7              SELECTED
    Tivozanib                    7              SELECTED


In [10]:
# CELL 8 — Build complexes with shared ligand PDB writer
COMPLEX_DIR.mkdir(parents=True, exist_ok=True)
if not RECEPTOR_PDB.exists():
    raise FileNotFoundError(f"Receptor PDB required for complex creation: {RECEPTOR_PDB}")
receptor_lines = [L.rstrip("\n") for L in RECEPTOR_PDB.read_text(errors="replace").splitlines()
                  if L.startswith("ATOM")]
if not receptor_lines:
    raise RuntimeError(f"No ATOM records found in receptor PDB: {RECEPTOR_PDB}")

complex_rows = []
for _, row in representative_df.iterrows():
    name = str(row["Name"])
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    if row["Representative_status"] != "SELECTED":
        complex_rows.append({"Name": name, "Representative_seed": row.get("Representative_seed"),
                             "Pose_consistency": row.get("Pose_consistency"),
                             "Complex_PDB": "", "Ligand_atoms_in_complex": 0,
                             "Status": "SKIPPED_NO_REPRESENTATIVE_POSE"})
        continue
    lig_pdb = COMPLEX_DIR / f"{safe}_ligand.pdb"
    cplx = COMPLEX_DIR / f"{safe}_complex.pdb"
    try:
        n = write_ligand_pdb(row["Representative_pose_PDBQT"], lig_pdb)
        if n <= 0 or not lig_pdb.exists():
            raise RuntimeError("Ligand PDB writer produced no atom records")
        lig_lines = [L.rstrip("\n") for L in lig_pdb.read_text(errors="replace").splitlines()
                     if L.startswith(("ATOM", "HETATM"))]
        if not lig_lines:
            raise RuntimeError("No ligand ATOM/HETATM records in generated PDB")
        cplx.write_text("\n".join(receptor_lines + ["TER"] + lig_lines + ["TER", "END"]) + "\n")
        complex_rows.append({"Name": name, "Representative_seed": row["Representative_seed"],
                             "Pose_consistency": row["Pose_consistency"],
                             "Complex_PDB": str(cplx), "Ligand_atoms_in_complex": n,
                             "Status": "PASS"})
    except Exception as exc:
        complex_rows.append({"Name": name, "Representative_seed": row.get("Representative_seed"),
                             "Pose_consistency": row.get("Pose_consistency"),
                             "Complex_PDB": str(cplx), "Ligand_atoms_in_complex": 0,
                             "Status": f"FAILED: {exc}"})
complex_df = pd.DataFrame(complex_rows)
complex_df.to_csv(COMPLEX_DIR / "complex_qc.csv", index=False)
print(complex_df[["Name", "Ligand_atoms_in_complex", "Status"]].to_string(index=False))


         Name  Ligand_atoms_in_complex Status
 Cabozantinib                       37   PASS
   Lenvatinib                       30   PASS
  Quizartinib                       40   PASS
Selpercatinib                       39   PASS
    Sorafenib                       32   PASS
    Tivozanib                       32   PASS


In [11]:
# CELL 9 — PLIP per candidate (optional dependency; output filename unchanged)
PLIP_DIR.mkdir(parents=True, exist_ok=True)
plip_rows = []
if not PLIP_AVAILABLE or not PLIP_EXE:
    for _, row in complex_df.iterrows():
        if row.get("Status") == "PASS":
            name = str(row["Name"])
            plip_rows.append({"Name": name, "PLIP_output_dir": str(PLIP_DIR / re.sub(r"[^A-Za-z0-9_.-]+", "_", name)),
                              "Return_code": np.nan, "Runtime_sec": np.nan,
                              "XML_found": False, "Status": "SKIPPED_NO_PLIP",
                              "QC_note": "PLIP executable not found on PATH; docking and complex generation completed."})
else:
    for _, row in complex_df.iterrows():
        if row.get("Status") != "PASS":
            continue
        name = str(row["Name"])
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
        out_dir = PLIP_DIR / safe
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = [str(PLIP_EXE), "-f", str(row["Complex_PDB"]), "-o", str(out_dir),
               "-x", "-t", "--name", safe]
        t0 = time.time()
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            elapsed = time.time() - t0
            xmls = list(out_dir.glob("*.xml"))
            status = "PASS" if (r.returncode == 0 and xmls) else "FAILED"
            note = "" if status == "PASS" else ((r.stderr or r.stdout or "No XML generated")[-1000:])
            plip_rows.append({"Name": name, "PLIP_output_dir": str(out_dir),
                              "Return_code": r.returncode, "Runtime_sec": round(elapsed, 3),
                              "XML_found": bool(xmls), "Status": status, "QC_note": note})
        except Exception as exc:
            plip_rows.append({"Name": name, "PLIP_output_dir": str(out_dir),
                              "Return_code": np.nan, "Runtime_sec": round(time.time()-t0, 3),
                              "XML_found": False, "Status": "FAILED",
                              "QC_note": str(exc)})
plip_qc_df = pd.DataFrame(plip_rows, columns=["Name", "PLIP_output_dir", "Return_code",
                                              "Runtime_sec", "XML_found", "Status", "QC_note"])
plip_qc_df.to_csv(PLIP_DIR / "plip_execution_qc.csv", index=False)
if plip_qc_df.empty:
    print("No complexes were eligible for PLIP. Empty QC table saved.")
else:
    print(plip_qc_df[["Name", "Status", "XML_found"]].to_string(index=False))


         Name Status  XML_found
 Cabozantinib   PASS       True
   Lenvatinib   PASS       True
  Quizartinib   PASS       True
Selpercatinib   PASS       True
    Sorafenib   PASS       True
    Tivozanib   PASS       True


In [12]:
# CELL 10 — Parse PLIP XML, build detailed + summary tables
import xml.etree.ElementTree as ET
SECTION_MAP = {"hydrophobic_interactions": "Hydrophobic", "hydrogen_bonds": "H-bond",
               "water_bridges": "Water bridge", "salt_bridges": "Salt bridge",
               "pi_stacks": "Pi-stack", "pistacking": "Pi-stack",
               "pi_cation_interactions": "Pi-cation", "pication": "Pi-cation",
               "halogen_bonds": "Halogen", "metal_complexes": "Metal"}

def local(t): return t.split("}")[-1]
def get_text(el, names):
    wanted = {n.lower() for n in names}
    for c in el.iter():
        if local(c.tag).lower() in wanted and c.text and c.text.strip():
            return c.text.strip()
    return ""
def get_float(el, names):
    try: return float(get_text(el, names))
    except (TypeError, ValueError): return np.nan

detail_rows = []
for _, row in plip_qc_df.iterrows():
    if row["Status"] != "PASS": continue
    xmls = list(Path(row["PLIP_output_dir"]).glob("*.xml"))
    if not xmls: continue
    root = ET.parse(xmls[0]).getroot()
    for section in root.iter():
        tag = local(section.tag)
        if tag not in SECTION_MAP: continue
        itype = SECTION_MAP[tag]
        for child in section:
            rec = {"Name": row["Name"], "Interaction_type": itype,
                   "Residue_number": get_text(child, ["resnr", "resnum", "residue_number"]),
                   "Residue_name": get_text(child, ["restype", "resname", "residue_name"]),
                   "Chain": get_text(child, ["reschain", "chain", "chain_id"]),
                   "Distance_A": get_float(child, ["dist", "distance", "dist_h-a", "dist_d-a"])}
            if any([rec["Residue_number"], rec["Residue_name"], np.isfinite(rec["Distance_A"])]):
                detail_rows.append(rec)
detail_df = pd.DataFrame(detail_rows, columns=["Name", "Interaction_type", "Residue_number", "Residue_name", "Chain", "Distance_A"])
detail_df.to_csv(PLIP_DIR / "plip_interactions_detailed.csv", index=False)

ITYPES = ["H-bond", "Hydrophobic", "Salt bridge", "Pi-stack", "Pi-cation", "Halogen", "Water bridge"]
if len(detail_df) > 0:
    counts = detail_df.groupby(["Name", "Interaction_type"]).size().unstack(fill_value=0)
    for t in ITYPES:
        if t not in counts.columns: counts[t] = 0
    summary_df = counts[ITYPES].reset_index()
    summary_df["Total"] = summary_df[ITYPES].sum(axis=1)
else:
    summary_df = pd.DataFrame(columns=["Name"] + ITYPES + ["Total"])
summary_df.to_csv(PLIP_DIR / "plip_interactions_summary.csv", index=False)
print(summary_df.to_string(index=False))

         Name  H-bond  Hydrophobic  Salt bridge  Pi-stack  Pi-cation  Halogen  Water bridge  Total
 Cabozantinib       3            9            0         0          0        0             0     12
   Lenvatinib       1            9            0         1          0        0             0     11
  Quizartinib       4            8            0         0          0        0             0     12
Selpercatinib       1            4            1         0          0        0             0      6
    Sorafenib       3            8            0         1          0        1             0     13
    Tivozanib       1            8            0         1          0        0             0     10


In [13]:
# CELL 11 — Integrated analysis with two-tier gate
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------
# 1. Per-candidate aggregate from the successful docking runs
# ---------------------------------------------------------------
aff = (
    successful
    .groupby("Name")["Affinity_kcal_mol"]
    .agg(Best_affinity_kcal_mol="min",
         Mean_affinity_kcal_mol="mean",
         SD_affinity_kcal_mol="std",
         Docking_runs="count")
    .reset_index()
)

# ---------------------------------------------------------------
# 2. Score of the representative seed (the pose PLIP actually saw)
# ---------------------------------------------------------------
rep_scores = []
for _, row in representative_df.iterrows():
    name = str(row["Name"])
    seed = row.get("Representative_seed")
    if pd.isna(seed):
        rep_scores.append({"Name": name,
                           "Representative_seed_affinity_kcal_mol": np.nan})
        continue
    hit = successful[
        (successful["Name"].astype(str) == name)
        & (successful["Seed"].astype(str) == str(int(seed)))
    ]
    val = float(hit["Affinity_kcal_mol"].iloc[0]) if not hit.empty else np.nan
    rep_scores.append({"Name": name,
                       "Representative_seed_affinity_kcal_mol": val})
rep_score_df = pd.DataFrame(rep_scores)

# ---------------------------------------------------------------
# 3. Merge the analysis layers
# ---------------------------------------------------------------
def _cols(df, wanted):
    return [c for c in wanted if c in df.columns]

integrated = (
    aff
    .merge(pose_repro_df[_cols(pose_repro_df,
                               ["Name", "Max_pairwise_RMSD_A",
                                "Mean_pairwise_RMSD_A",
                                "Internal_conformer_RMSD_max_A",
                                "Pose_consistency"])],
           on="Name", how="left")
    .merge(representative_df[_cols(representative_df,
                                   ["Name", "Representative_seed",
                                    "Representative_pose_PDBQT"])],
           on="Name", how="left")
    .merge(rep_score_df, on="Name", how="left")
    .merge(complex_df[_cols(complex_df,
                            ["Name", "Complex_PDB", "Status"])]
             .rename(columns={"Status": "Complex_QC"}),
           on="Name", how="left")
    .merge(plip_qc_df[_cols(plip_qc_df, ["Name", "Status"])]
             .rename(columns={"Status": "PLIP_QC"}),
           on="Name", how="left")
    .merge(summary_df.rename(columns={"Total": "Total_interactions"}),
           on="Name", how="left")
)

# ---------------------------------------------------------------
# 4. Priority tier — pose-consistency gate, then best Vina score
# ---------------------------------------------------------------
def tier(row):
    if str(row.get("Pose_consistency", "")).upper() != "YES":
        return "Tier-C (pose unstable)"
    v = row.get("Best_affinity_kcal_mol", np.nan)
    if pd.isna(v):
        return "Tier-C (no score)"
    if v <= -10.0:
        return "Tier-A"
    if v <= -8.0:
        return "Tier-B"
    return "Tier-C"

integrated["Priority_tier"] = integrated.apply(tier, axis=1)

# ---------------------------------------------------------------
# 5. Overall QC (pipeline ran cleanly, not a scientific verdict)
# ---------------------------------------------------------------
integrated["Overall_QC"] = integrated.apply(
    lambda r: "PASS"
    if (str(r.get("Complex_QC", "")).upper() == "PASS"
        and str(r.get("PLIP_QC", "")).upper() == "PASS")
    else "CHECK",
    axis=1,
)

# ---------------------------------------------------------------
# 6. Flag when the tier score and the representative seed score
#    disagree meaningfully (>= 1 kcal/mol worse on the PLIP pose)
# ---------------------------------------------------------------
def _score_gap_flag(row):
    best = row.get("Best_affinity_kcal_mol", np.nan)
    rep = row.get("Representative_seed_affinity_kcal_mol", np.nan)
    if pd.isna(best) or pd.isna(rep):
        return "n/a"
    gap = rep - best  # positive = representative pose is worse
    if gap >= 1.0:
        return f"representative seed {gap:+.2f} kcal/mol vs best"
    return ""

integrated["Tier_vs_PLIP_pose_gap"] = integrated.apply(_score_gap_flag, axis=1)

# ---------------------------------------------------------------
# 7. Sort and save
# ---------------------------------------------------------------
tier_order = {
    "Tier-A": 1,
    "Tier-B": 2,
    "Tier-C": 3,
    "Tier-C (pose unstable)": 4,
    "Tier-C (no score)": 5,
}
integrated["_tier_order"] = integrated["Priority_tier"].map(tier_order).fillna(9)
integrated = (
    integrated
    .sort_values(["_tier_order", "Best_affinity_kcal_mol"])
    .drop(columns=["_tier_order"])
    .reset_index(drop=True)
)

integrated.to_csv(REPORT_DIR / "integrated_final_analysis.csv", index=False)

# ---------------------------------------------------------------
# 8. Console summary
# ---------------------------------------------------------------
cols = [c for c in [
    "Name",
    "Best_affinity_kcal_mol",
    "Representative_seed_affinity_kcal_mol",
    "Mean_affinity_kcal_mol",
    "SD_affinity_kcal_mol",
    "Docking_runs",
    "Max_pairwise_RMSD_A",
    "Pose_consistency",
    "Representative_seed",
    "Total_interactions",
    "Priority_tier",
    "Overall_QC",
    "Tier_vs_PLIP_pose_gap",
] if c in integrated.columns]

print(integrated[cols].to_string(index=False))

# ---------------------------------------------------------------
# 9. Explicit heads-up on any disagreement
# ---------------------------------------------------------------
flagged = integrated[integrated["Tier_vs_PLIP_pose_gap"].astype(str).str.len() > 0]
if not flagged.empty:
    print("\n⚠ Representative seed score differs from best by >= 1 kcal/mol:")
    print(flagged[["Name",
                   "Best_affinity_kcal_mol",
                   "Representative_seed_affinity_kcal_mol",
                   "Tier_vs_PLIP_pose_gap"]].to_string(index=False))
    print("\n  The tier reflects the best pose; the PLIP contacts describe the")
    print("  representative seed. Report both when these differ.")
else:
    print("\nBest score and representative seed score agree within 1 kcal/mol for all rows.")

         Name  Best_affinity_kcal_mol  Representative_seed_affinity_kcal_mol  Mean_affinity_kcal_mol  SD_affinity_kcal_mol  Docking_runs  Max_pairwise_RMSD_A Pose_consistency  Representative_seed  Total_interactions          Priority_tier Overall_QC Tier_vs_PLIP_pose_gap
    Sorafenib                 -10.980                                -10.980              -10.950000              0.043589             3             0.072330              YES                  7.0                13.0                 Tier-A       PASS                      
   Lenvatinib                 -10.230                                -10.220              -10.220000              0.010000             3             0.091927              YES               2026.0                11.0                 Tier-A       PASS                      
    Tivozanib                 -10.220                                -10.220              -10.213333              0.005774             3             0.069104              YES          

In [14]:
# CELL 12 — Metadata and reproducibility record
# Depends on cells 1–11 in this kernel:
#   CANDIDATE_FILE, RECEPTOR_PDBQT, BOX, VINA, PLIP_EXE, DOCKING_DIR,
#   REPORT_DIR, EXHAUSTIVENESS, NUM_MODES, ENERGY_RANGE, CPU, SEEDS,
#   POSE_CONSISTENCY_THRESHOLD, integrated, sha256_file, pd, np, json,
#   subprocess, sys, datetime, timezone.
#
# Also uses PLIP_AVAILABLE if defined.
# Control-docking summary is read from:
#   DOCKING_DIR / "controls" / "control_roc_summary.json"

from pathlib import Path


# ---------------------------------------------------------------
# 1. Helpers
# ---------------------------------------------------------------

def tool_version(exe, primary, fallback=None, max_len=2000):
    """Return tool version/help text without interrupting the workflow."""
    if exe is None:
        return None

    try:
        result = subprocess.run(
            [str(exe)] + list(primary),
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )

        output = (result.stdout or result.stderr or "").strip()

        if output:
            return output[:max_len]

    except Exception:
        pass

    if fallback is not None:
        try:
            result = subprocess.run(
                [str(exe)] + list(fallback),
                capture_output=True,
                text=True,
                timeout=30,
                check=False,
            )

            output = (result.stdout or result.stderr or "").strip()

            if output:
                return output[:max_len]

        except Exception as exc:
            return f"unavailable: {exc}"

    return None


def python_module_version(name):
    """Return an installed Python package version where available."""
    try:
        module = __import__(name)
        return str(getattr(module, "__version__", "unknown"))
    except Exception:
        return "not installed"


def safe_display_value(value, max_len=60):
    """
    Convert any metadata value to a safe, compact string.

    Handles strings, Booleans, numbers, lists, dictionaries, and None.
    In particular, avoids calling splitlines() on a Boolean.
    """
    if value is None:
        return "n/a"

    text_value = str(value).strip()

    if not text_value:
        return "n/a"

    first_line = text_value.splitlines()[0].strip()
    return first_line[:max_len] if first_line else "n/a"


def json_safe_dataframe_records(dataframe, columns):
    """Convert selected DataFrame columns to JSON-friendly records."""
    if not columns:
        return []

    clean_df = dataframe[columns].copy()
    clean_df = clean_df.astype(object).where(pd.notna(clean_df), None)

    return clean_df.to_dict(orient="records")


def is_valid_two_value_ci(value):
    """Check whether a confidence interval contains two numeric values."""
    if not isinstance(value, (list, tuple)) or len(value) != 2:
        return False

    try:
        lower, upper = float(value[0]), float(value[1])
        return (
            np.isfinite(lower)
            and np.isfinite(upper)
            and lower <= upper
        )
    except (TypeError, ValueError):
        return False


# ---------------------------------------------------------------
# 2. Validate essential inputs
# ---------------------------------------------------------------

if not isinstance(integrated, pd.DataFrame):
    raise TypeError(
        "'integrated' must be a pandas DataFrame. "
        "Run the integrated analysis cell before Cell 12."
    )

if integrated.empty:
    raise ValueError(
        "'integrated' contains no candidate rows. "
        "Check the integrated analysis output before writing metadata."
    )

required_metadata_columns = ["Priority_tier", "Overall_QC"]

missing_columns = [
    column
    for column in required_metadata_columns
    if column not in integrated.columns
]

if missing_columns:
    raise KeyError(
        "Required columns are missing from 'integrated': "
        + ", ".join(missing_columns)
    )

REPORT_DIR = Path(REPORT_DIR)
DOCKING_DIR = Path(DOCKING_DIR)

REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------
# 3. Read control-docking summary, if available
# ---------------------------------------------------------------

control_dir = DOCKING_DIR / "controls"
control_json = control_dir / "control_roc_summary.json"

control_summary = None

if control_json.exists():
    try:
        with open(control_json, "r", encoding="utf-8") as handle:
            control_summary = json.load(handle)

        if not isinstance(control_summary, dict):
            control_summary = {
                "error": "Control summary JSON is not a dictionary."
            }

    except Exception as exc:
        control_summary = {"error": str(exc)}

control_ran = bool(
    isinstance(control_summary, dict)
    and "error" not in control_summary
)

control_reduced_scope = bool(
    control_ran
    and control_summary.get("reduced_scope", True)
)

if not control_ran:
    control_scope = "not_run"
elif control_reduced_scope:
    control_scope = "reduced"
else:
    control_scope = "full"


# ---------------------------------------------------------------
# 4. Flag disagreement between representative pose and best score
# ---------------------------------------------------------------

flagged_rows = []

if "Tier_vs_PLIP_pose_gap" in integrated.columns:
    gap_values = integrated["Tier_vs_PLIP_pose_gap"]

    # Exclude missing values and empty strings.
    has_gap = (
        gap_values.notna()
        & gap_values.astype(str).str.strip().ne("")
        & gap_values.astype(str).str.lower().ne("nan")
    )

    flagged_columns = [
        column
        for column in [
            "Name",
            "Best_affinity_kcal_mol",
            "Representative_seed_affinity_kcal_mol",
            "Tier_vs_PLIP_pose_gap",
        ]
        if column in integrated.columns
    ]

    flagged_rows = json_safe_dataframe_records(
        integrated.loc[has_gap],
        flagged_columns,
    )


# ---------------------------------------------------------------
# 5. Compact per-candidate summary
# ---------------------------------------------------------------

preferred_candidate_columns = [
    "Name",
    "Best_affinity_kcal_mol",
    "Representative_seed_affinity_kcal_mol",
    "Mean_affinity_kcal_mol",
    "SD_affinity_kcal_mol",
    "Docking_runs",
    "Max_pairwise_RMSD_A",
    "Pose_consistency",
    "Representative_seed",
    "Total_interactions",
    "Priority_tier",
    "Overall_QC",
]

per_candidate_cols = [
    column
    for column in preferred_candidate_columns
    if column in integrated.columns
]

per_candidate = json_safe_dataframe_records(
    integrated,
    per_candidate_cols,
)


# ---------------------------------------------------------------
# 6. Priority-rule calibration assessment
#
# Calibration is marked true only if:
#   1. A valid ROC AUC 95% CI is available;
#   2. Its lower bound is >= 0.6; and
#   3. The control run was full scope.
# ---------------------------------------------------------------

auc_ci95 = (
    control_summary.get("auc_ci95")
    if control_ran
    else None
)

_ctrl_has_ci = bool(
    control_ran and is_valid_two_value_ci(auc_ci95)
)

_ctrl_ci_ok = bool(
    _ctrl_has_ci and float(auc_ci95[0]) >= 0.6
)

_ctrl_full_scope = bool(
    control_ran and not control_reduced_scope
)

_calibrated = bool(
    _ctrl_has_ci
    and _ctrl_ci_ok
    and _ctrl_full_scope
)

priority_rule = {
    "tier_a": "pose-consistent and best Vina <= -10 kcal/mol",
    "tier_b": "pose-consistent and best Vina <= -8 kcal/mol",
    "tier_c": "otherwise",
    "calibrated": _calibrated,
    "calibration_note": (
        "Thresholds are user-set. 'calibrated' is true only when the "
        "control-docking ROC 95% CI lower bound is >= 0.6 and the "
        "control run was full scope (20 actives, 200 decoys, 3 seeds). "
        "Even then, the specific cut-offs are not empirically optimised."
    ),
}


# ---------------------------------------------------------------
# 7. Tier caveat, conditional on control-docking scope
# ---------------------------------------------------------------

if control_ran and control_reduced_scope:
    caveats_tier = (
        "Tier thresholds (-10 and -8 kcal/mol) are user-set and were NOT "
        "calibrated against DDR1 actives or decoys: the control-docking ROC "
        "was run in reduced scope. Re-run the controls at full scope "
        "(CONTROL_SEEDS = SEEDS, CONTROL_N_DECOYS = 200) before publishing "
        "any claim that the tiers rank known DDR1 binders above "
        "property-matched decoys."
    )

elif control_ran:
    caveats_tier = (
        "Tier thresholds (-10 and -8 kcal/mol) are user-set. The control "
        "ROC was run at full scope; see 'control_docking' for AUC and CI. "
        "Even at full scope, the specific cut-offs are not empirically "
        "optimised."
    )

else:
    caveats_tier = (
        "Tier thresholds (-10 and -8 kcal/mol) are user-set and no "
        "valid control-docking ROC summary was available in this session. "
        "The tiers should be treated as a ranking heuristic only."
    )


# ---------------------------------------------------------------
# 8. Collect software versions
# ---------------------------------------------------------------

plip_available_value = bool(globals().get("PLIP_AVAILABLE", False))

tool_versions = {
    "python": sys.version.split()[0],
    "rdkit": python_module_version("rdkit"),
    "meeko": python_module_version("meeko"),
    "pandas": python_module_version("pandas"),
    "numpy": python_module_version("numpy"),
    "vina": tool_version(
        VINA,
        ["--version"],
        fallback=["--help"],
    ),
    "plip_available": plip_available_value,
    "plip": tool_version(
        PLIP_EXE,
        ["--version"],
        fallback=["--help"],
    ),
}


# ---------------------------------------------------------------
# 9. Build metadata record
# ---------------------------------------------------------------

meta = {
    "analysis_date_utc": datetime.now(timezone.utc).isoformat(),
    "pipeline": "DDR1_candidate_docking",

    # Inputs and fingerprints
    "inputs": {
        "candidate_file": str(CANDIDATE_FILE),
        "candidate_file_sha256": sha256_file(CANDIDATE_FILE),
        "receptor_pdbqt": str(RECEPTOR_PDBQT),
        "receptor_sha256": sha256_file(RECEPTOR_PDBQT),
        "receptor_source": (
            "Notebook 2 output, verified by SHA256 match"
        ),
        "box": BOX,
    },

    # Software versions
    "tool_versions": tool_versions,

    "plip_version_note": (
        "Not all PLIP builds support --version. When the short flag "
        "fails, the --help output is stored as the fallback. "
        "'plip_available' is a Boolean availability flag, not a version."
    ),

    # Docking and analysis protocol
    "protocol": {
        "box_center_source": (
            "derived in Notebook 2 from the 1JI centroid"
        ),
        "exhaustiveness": EXHAUSTIVENESS,
        "num_modes": NUM_MODES,
        "energy_range_kcal_mol": ENERGY_RANGE,
        "cpu": CPU,
        "seeds": SEEDS,
        "pose_consistency_threshold_A": POSE_CONSISTENCY_THRESHOLD,

        "rmsd_method": (
            "Direct heavy-atom RMSD between poses in the receptor frame. "
            "No superposition is performed; translation and rotation are "
            "part of the pose. RMSD is symmetry-corrected by minimizing "
            "over graph-valid heavy-atom automorphisms of the reference "
            "ligand. Atom correspondence is derived from Meeko SMILES IDX "
            "metadata and a stereochemistry-aware full-graph mapping. "
            "Unresolved mappings receive no numeric RMSD."
        ),

        "representative_pose_rule": (
            "Most central of the three seed poses, defined as the pose "
            "with the lowest mean pairwise direct RMSD to the other two."
        ),

        "plip_input_pose": (
            "The representative seed's pose, not necessarily the "
            "best-scoring pose. Rows where the two differ by >= 1 kcal/mol "
            "are listed under 'representative_vs_best_disagreements'."
        ),
    },

    # Prioritization rule
    "priority_rule": priority_rule,

    # Control docking
    "control_docking": (
        control_summary
        if control_ran
        else {
            "status": "not run or valid summary unavailable in this session",
            "expected_file": str(control_json),
            "details": (
                control_summary.get("error")
                if isinstance(control_summary, dict)
                else None
            ),
        }
    ),

    "control_docking_scope": control_scope,

    # Integrated results
    "n_candidates": int(len(integrated)),

    "tier_counts": {
        str(key): int(value)
        for key, value in
        integrated["Priority_tier"].value_counts(dropna=False).items()
    },

    "overall_qc_counts": {
        str(key): int(value)
        for key, value in
        integrated["Overall_QC"].value_counts(dropna=False).items()
    },

    "per_candidate_summary": per_candidate,

    # Disagreements between representative pose and best score
    "representative_vs_best_disagreements": flagged_rows,

    # Scientific caveats
    "caveats": [
        "Vina scores are not Delta-G values.",

        "PLIP contacts are geometric predictions, not measured affinities.",

        (
            "Only one receptor structure (5SB1) was used; DFG-state "
            "and receptor flexibility were not explored."
        ),

        (
            "Ligands were docked in their neutral form. pH 7.4 protonation "
            "states were not enumerated. Amines, amidines, and carboxylic "
            "acids are particularly sensitive to protonation state."
        ),

        (
            "Pose RMSD is symmetry-corrected using graph automorphisms and "
            "calculated directly in the receptor coordinate frame without "
            "superposition. Atom-order and atom-type checks must pass; "
            "unresolved mappings are not assigned a numeric RMSD. Review "
            "any UNRESOLVED candidate before interpreting its tier."
        ),

        caveats_tier,

        (
            "Priority_tier is a two-factor gate based on pose stability "
            "and best Vina score. Interaction counts and pose reproducibility "
            "are reported as context and are NOT part of the tier decision."
        ),

        (
            "Overall_QC reflects whether the pipeline ran cleanly, not "
            "whether the pose is biologically trustworthy. Interpret it "
            "together with Pose_consistency and Priority_tier."
        ),
    ],
}


# ---------------------------------------------------------------
# 10. Persist metadata
# ---------------------------------------------------------------

metadata_file = REPORT_DIR / "software_and_protocol_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as handle:
    json.dump(
        meta,
        handle,
        indent=2,
        default=str,
        ensure_ascii=False,
    )


# ---------------------------------------------------------------
# 11. Console summary
# ---------------------------------------------------------------

print(f"Integrated results : {REPORT_DIR / 'integrated_final_analysis.csv'}")
print(f"Metadata           : {metadata_file}")
print()

print("Tool versions:")

for key, value in meta["tool_versions"].items():
    value_short = safe_display_value(value)
    print(f"  {key:18s}: {value_short}")

print()
print(f"Candidates         : {meta['n_candidates']}")
print("Tier counts:")

for tier_name in [
    "Tier-A",
    "Tier-B",
    "Tier-C",
    "Tier-C (pose unstable)",
    "Tier-C (no score)",
]:
    count = meta["tier_counts"].get(tier_name, 0)

    if count:
        print(f"  {tier_name:26s}: {count}")

print()

if control_ran:
    print("Control docking (actives vs matched decoys):")
    print(f"  Scope         : {control_scope.upper()}")
    print(f"  Actives       : {control_summary.get('n_actives', 'n/a')}")
    print(f"  Decoys        : {control_summary.get('n_decoys', 'n/a')}")

    seeds_used = control_summary.get(
        "seeds_used",
        [control_summary.get("seeds_per_compound", "n/a")],
    )
    print(f"  Seeds/compound: {seeds_used}")

    auc = control_summary.get("auc")

    try:
        print(f"  AUC           : {float(auc):.3f}")
    except (TypeError, ValueError):
        print(f"  AUC           : {auc if auc is not None else 'n/a'}")

    if _ctrl_has_ci:
        print(
            f"  95% CI        : "
            f"{float(auc_ci95[0]):.3f}–{float(auc_ci95[1]):.3f}"
        )
    else:
        print("  95% CI        : n/a")

    if "ef_5pct" in control_summary:
        try:
            print(f"  EF @ 5%       : {float(control_summary['ef_5pct']):.2f}")
        except (TypeError, ValueError):
            print(f"  EF @ 5%       : {control_summary['ef_5pct']}")

    if priority_rule["calibrated"]:
        print("  Tiers         : marked as calibrated.")

    elif control_reduced_scope:
        print("  Tiers         : NOT calibrated (reduced-scope control run).")
        print("                  Re-run at full scope before publishing tier claims.")

    elif _ctrl_has_ci and not _ctrl_ci_ok:
        print("  Tiers         : NOT calibrated (95% CI lower bound < 0.6).")
        print("                  The protocol may not separate actives from decoys.")

    else:
        print("  Tiers         : NOT calibrated (valid CI unavailable).")

else:
    print("Control docking: not run or valid summary unavailable.")
    print("  The tier thresholds remain uncalibrated.")

if flagged_rows:
    print()
    print(
        "WARNING: Representative seed differs from best score "
        "by >= 1 kcal/mol:"
    )

    for row in flagged_rows:
        name = row.get("Name", "Unknown")
        best = row.get("Best_affinity_kcal_mol")
        representative = row.get("Representative_seed_affinity_kcal_mol")
        gap = row.get("Tier_vs_PLIP_pose_gap")

        print(
            f"  {name}: best={safe_display_value(best)}, "
            f"PLIP pose={safe_display_value(representative)}, "
            f"gap={safe_display_value(gap)}"
        )

print()
print("Metadata and reproducibility record completed.")
print("Next: run 4-DDR1-VIZ.")

Integrated results : /content/drive/MyDrive/DDR1_Workflow/candidate_docking/MANUSCRIPT_GRADE_DDR1_ANALYSIS/reports/integrated_final_analysis.csv
Metadata           : /content/drive/MyDrive/DDR1_Workflow/candidate_docking/MANUSCRIPT_GRADE_DDR1_ANALYSIS/reports/software_and_protocol_metadata.json

Tool versions:
  python            : 3.13.16
  rdkit             : 2026.09.1
  meeko             : 0.8.0
  pandas            : 2.2.3
  numpy             : 2.1.3
  vina              : AutoDock Vina v1.2.5
  plip_available    : True
  plip              : usage: PLIP [-h] (-f INPUT [INPUT ...] | -i PDBID [PDBID ...

Candidates         : 7
Tier counts:
  Tier-A                    : 4
  Tier-B                    : 2
  Tier-C (pose unstable)    : 1

Control docking (actives vs matched decoys):
  Scope         : REDUCED
  Actives       : n/a
  Decoys        : n/a
  Seeds/compound: ['n/a']
  AUC           : 0.869
  95% CI        : 0.776–0.941
  Tiers         : NOT calibrated (reduced-scope control run)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
